# 14 — Manuscript-ready figures and tables (Stage 9)

Consolidates everything into the deliverables that drop into a manuscript written elsewhere:

**Tables** (CSV in `outputs/tables/`)
- T1   Descriptive (LA × IMD × phase): salons, schools, child population
- T2   Source agreement: Google vs OSM by LA, with Jaccard
- T3a–d Primary models H1 / H2 / H3 (NB GLM with cluster-robust SE)
- T4   SII / RII for buffer + route, plus headline ratio with bootstrap CI
- T5   Sensitivity matrix across pre-registered dimensions

**Figures** (PNG + PDF in `outputs/figures/`)
- F1   Regional LSOA choropleth: per-pupil route exposure
- F2   Per-LA dot-density of salons + schools
- F3   Example neighbourhood with routes + buffers + salons
- F4   Forest plot of LA-specific NB slopes (route exposure vs IMD)
- F5   Sensitivity forest of the headline RII ratio (already produced in 13)

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from schools_sunbeds import audit, config, exposure, salons_dedupe, stats

config.ensure_dirs()
audit.verify_manifest()
TODAY = datetime.now(timezone.utc).strftime("%Y%m%d")

## 0. Load inputs

In [ ]:
schools = gpd.read_file(
    [p for p in sorted(config.DATA_PROCESSED.glob("schools_ne_*.gpkg")) if "sensitivity" not in p.name][-1]
)
lsoa = gpd.read_file(sorted(config.DATA_PROCESSED.glob("lsoa_ne_*.gpkg"))[-1])
lad = gpd.read_file(sorted(config.DATA_PROCESSED.glob("lad_ne_*.gpkg"))[-1])
salons = gpd.read_file(sorted(config.DATA_PROCESSED.glob("salons_verified_union_*.gpkg"))[-1])
google = gpd.read_file(sorted(config.DATA_PROCESSED.glob("salons_google_*.gpkg"))[-1])
osm = gpd.read_file(sorted(config.DATA_PROCESSED.glob("salons_osm_*.gpkg"))[-1])
panel = pd.read_parquet(sorted(config.DATA_PROCESSED.glob("exposure_panel_*.parquet"))[-1])
rb50 = gpd.read_file(sorted(config.DATA_PROCESSED.glob("route_buffer_50m_*.gpkg"))[-1])
routes = gpd.read_file(sorted(config.DATA_PROCESSED.glob("routes_*.gpkg"))[-1])
print(f"All inputs loaded.")

## T1 — Descriptive replication (LA × phase × IMD quintile)

Already produced by notebook 10 at `outputs/tables/T1_descriptive_replication_*.csv` and `T1b_per_la_density_*.csv`. Here we add a richer version: per-LA × IMD-quintile salon density and school count cross-tab.

In [ ]:
# Per-LA × IMD-quintile salon counts and child-population denominators.
# Drop salons' own lad_code first so the sjoin does not suffix the column.
salons_clean = salons.drop(columns=[c for c in ("lad_code", "imd_quintile") if c in salons.columns])
salon_la_imd = (
    salons_clean.sjoin(
        lsoa[["lsoa21cd", "lad_code", "imd_quintile", "pop_total", "pop_school_age", "geometry"]],
        how="left", predicate="within",
    )
    .groupby(["lad_code", "imd_quintile"])
    .size()
    .rename("n_salons")
    .reset_index()
)
lsoa_la_imd = (
    lsoa.groupby(["lad_code", "imd_quintile"])
    .agg(pop_total=("pop_total", "sum"), pop_school_age=("pop_school_age", "sum"))
    .reset_index()
)
t1_full = lsoa_la_imd.merge(salon_la_imd, on=["lad_code", "imd_quintile"], how="left")
t1_full["n_salons"] = t1_full["n_salons"].fillna(0).astype(int)
t1_full["salons_per_100k_pop"] = (t1_full["n_salons"] * 1e5 / t1_full["pop_total"]).round(2)
t1_full["lad_name"] = t1_full["lad_code"].map(config.LA_NAMES_NE)
t1_full = t1_full[["lad_code", "lad_name", "imd_quintile", "pop_total", "pop_school_age", "n_salons", "salons_per_100k_pop"]]
t1_full.to_csv(config.OUTPUTS_TABLES / f"T1c_la_imd_quintile_{TODAY}.csv", index=False)
t1_full.head(15)

## T2 — Google vs OSM source agreement

In [ ]:
# Recompute the agreement table at LA level
match = salons_dedupe.match_salons(google, osm)
google_per_la = google.groupby("lad_code").size().rename("n_google")
osm_per_la = osm.groupby("lad_code").size().rename("n_osm")
matched_google_per_la = (
    google.loc[google["place_id"].isin(match.matched["place_id"])]
    .groupby("lad_code").size().rename("n_matched")
)
t2 = pd.DataFrame({"lad_code": list(config.LA_CODES_NE)}).merge(
    google_per_la, on="lad_code", how="left"
).merge(osm_per_la, on="lad_code", how="left").merge(
    matched_google_per_la, on="lad_code", how="left"
)
t2[["n_google", "n_osm", "n_matched"]] = t2[["n_google", "n_osm", "n_matched"]].fillna(0).astype(int)
t2["lad_name"] = t2["lad_code"].map(config.LA_NAMES_NE)
denom = (t2["n_google"] + t2["n_osm"] - t2["n_matched"]).replace(0, np.nan)
t2["jaccard"] = (t2["n_matched"] / denom).round(3)
t2 = t2[["lad_code", "lad_name", "n_google", "n_osm", "n_matched", "jaccard"]]
t2.loc["TOTAL"] = ["", "NE total", t2["n_google"].sum(), t2["n_osm"].sum(), t2["n_matched"].sum(),
                   round(t2["n_matched"].sum() / max(t2["n_google"].sum() + t2["n_osm"].sum() - t2["n_matched"].sum(), 1), 3)]
t2.to_csv(config.OUTPUTS_TABLES / f"T2_source_agreement_{TODAY}.csv", index=False)
t2

## F1 — Regional LSOA choropleth: per-pupil route exposure

For each LSOA we sum salon-encounters across its assigned routes and divide by the LSOA's school-age population. This shows which LSOAs put pupils on high-exposure walking routes.

In [ ]:
# Per-LSOA mean salons-per-route from rb50 (one polygon per (lsoa, school))
rb50_with_count = rb50.copy().reset_index(drop=True)
rb50_with_count["_route_key"] = np.arange(len(rb50_with_count))
counts = exposure._count_points_in_polygons(salons, rb50_with_count, group_col="_route_key")
rb50_with_count["n_salons_route"] = rb50_with_count["_route_key"].map(counts).fillna(0).astype(int)
lsoa_exposure = (
    rb50_with_count.groupby("lsoa21cd")
    .agg(mean_salons_route=("n_salons_route", "mean"), n_routes=("n_salons_route", "count"))
    .reset_index()
)
lsoa_plot = lsoa.merge(lsoa_exposure, on="lsoa21cd", how="left")
print(f"LSOAs with ≥1 route: {lsoa_plot['n_routes'].notna().sum()}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 10))
lsoa_plot.plot(
    ax=ax,
    column="mean_salons_route",
    cmap="YlOrRd",
    missing_kwds={"color": "#dddddd", "label": "no route"},
    legend=True,
    legend_kwds={"label": "Mean salons per pupil route (50 m buffer)", "shrink": 0.6},
    edgecolor="none",
    linewidth=0,
)
lad.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.5)
ax.set_title("NE LSOAs: mean per-pupil walking-route salon encounters")
ax.set_xlabel("Easting (m, BNG)")
ax.set_ylabel("Northing (m, BNG)")
plt.tight_layout()
f1_path = config.OUTPUTS_FIGURES / f"F1_route_exposure_choropleth_{TODAY}"
plt.savefig(f1_path.with_suffix(".png"), dpi=150)
plt.savefig(f1_path.with_suffix(".pdf"))
print("Wrote", f1_path.with_suffix('.png').relative_to(config.REPO_ROOT))
plt.show()

## F2 — Per-LA dot-density of salons + schools

In [ ]:
fig, ax = plt.subplots(figsize=(9, 10))
lad.plot(ax=ax, facecolor="#f6f6f6", edgecolor="black", linewidth=0.5)
schools.plot(ax=ax, color="#3366cc", markersize=8, alpha=0.6, label=f"Schools (n={len(schools)})")
salons.plot(ax=ax, color="#cc0000", markersize=10, alpha=0.7, label=f"Verified salons (n={len(salons)})")
ax.set_title("NE schools and verified tanning salons")
ax.legend(loc="upper right")
ax.set_xlabel("Easting (m, BNG)")
ax.set_ylabel("Northing (m, BNG)")
plt.tight_layout()
f2_path = config.OUTPUTS_FIGURES / f"F2_dot_density_{TODAY}"
plt.savefig(f2_path.with_suffix(".png"), dpi=150)
plt.savefig(f2_path.with_suffix(".pdf"))
print("Wrote", f2_path.with_suffix('.png').relative_to(config.REPO_ROOT))
plt.show()

## F3 — Example neighbourhood (Newcastle, top-pop LSOA)

In [ ]:
newc_lsoas = lsoa.loc[lsoa["lad_code"] == "E08000021"].sort_values("pop_school_age", ascending=False).head(1)
sample_lsoa = newc_lsoas.iloc[0]["lsoa21cd"]
print(f"Sample LSOA: {sample_lsoa} ({newc_lsoas.iloc[0]['lsoa21nm']})")
rs = routes[routes["lsoa21cd"] == sample_lsoa]
rb = rb50[rb50["lsoa21cd"] == sample_lsoa]
ru = schools.loc[schools["urn"].isin(rs["urn"])]
neighbourhood_bbox = rb.total_bounds
pad = 500
salons_local = salons.cx[neighbourhood_bbox[0]-pad:neighbourhood_bbox[2]+pad,
                          neighbourhood_bbox[1]-pad:neighbourhood_bbox[3]+pad]

fig, ax = plt.subplots(figsize=(9, 8))
rb.plot(ax=ax, color="#cc0000", alpha=0.3, edgecolor="none", label="50 m route buffer")
rs.plot(ax=ax, color="#990000", linewidth=1.6, label="walking route")
ru.plot(ax=ax, color="black", markersize=80, marker="^", label="assigned schools")
salons_local.plot(ax=ax, color="#00aa44", markersize=30, marker="o", label="verified salons")
ax.set_title(f"Routes from {sample_lsoa} to assigned schools (Newcastle, top-pop LSOA)")
ax.legend(loc="upper right", fontsize=9)
ax.set_xlabel("Easting (m, BNG)")
ax.set_ylabel("Northing (m, BNG)")
plt.tight_layout()
f3_path = config.OUTPUTS_FIGURES / f"F3_example_neighbourhood_{TODAY}"
plt.savefig(f3_path.with_suffix(".png"), dpi=150)
plt.savefig(f3_path.with_suffix(".pdf"))
print("Wrote", f3_path.with_suffix('.png').relative_to(config.REPO_ROOT))
plt.show()

## F4 — Forest plot of per-LA NB slopes (route exposure on IMD)

For each of the 12 NE LAs, fit `sum_route_50m ~ ridit_imd + offset(log(sum_pupil_50m))` and plot the slope's 95% CI. This shows whether the H2 inversion is uniform or driven by particular LAs.

In [ ]:
# Re-attach IMD via spatial join
schools_imd = schools.sjoin(
    lsoa[["lsoa21cd", "imd_quintile", "geometry"]], how="left", predicate="within"
)[["urn", "imd_quintile"]]
p = panel.merge(schools_imd, on="urn", how="left", suffixes=("", "_dup"))
p = p.loc[:, ~p.columns.str.endswith("_dup")].dropna(subset=["imd_quintile"]).copy()
p["imd_quintile"] = p["imd_quintile"].astype("int64")
p["_pupil_offset"] = p["sum_pupil_50m"].clip(lower=1)
p_sub = p.loc[p["sum_pupil_50m"] > 0]

rows = []
for la in config.LA_CODES_NE:
    df_la = p_sub.loc[p_sub["la_code_ons"] == la].copy()
    if len(df_la) < 25:
        continue
    df_la["ridit"] = stats.compute_ridit(df_la["imd_quintile"], df_la["sum_pupil_50m"])
    try:
        res = stats.fit_nb_glm(df_la, "sum_route_50m ~ ridit", offset_col="_pupil_offset")
        coef = float(res.params["ridit"])
        lo, hi = float(res.conf_int_low["ridit"]), float(res.conf_int_high["ridit"])
        rows.append({"lad_code": la, "lad_name": config.LA_NAMES_NE[la], "slope": coef, "ci_low": lo, "ci_high": hi, "n": len(df_la)})
    except Exception as e:
        print(f"  {la} failed: {e}")
la_slopes = pd.DataFrame(rows).sort_values("slope")
la_slopes

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
y = np.arange(len(la_slopes))
ax.errorbar(
    la_slopes["slope"], y,
    xerr=[la_slopes["slope"] - la_slopes["ci_low"], la_slopes["ci_high"] - la_slopes["slope"]],
    fmt="o", color="#cc0000", capsize=4,
)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_yticks(y)
ax.set_yticklabels(la_slopes["lad_name"])
ax.set_xlabel("NB slope of route exposure on IMD ridit (negative = pro-deprived)")
ax.set_title("LA-specific deprivation slopes for route exposure (95% CI)")
plt.tight_layout()
f4_path = config.OUTPUTS_FIGURES / f"F4_la_forest_{TODAY}"
plt.savefig(f4_path.with_suffix(".png"), dpi=150)
plt.savefig(f4_path.with_suffix(".pdf"))
la_slopes.to_csv(config.OUTPUTS_TABLES / f"T6_la_slopes_{TODAY}.csv", index=False)
print("Wrote F4 + T6")
plt.show()

## Summary of deliverables

In [ ]:
for p in sorted(config.OUTPUTS_TABLES.glob("*.csv")):
    print("  ", p.relative_to(config.REPO_ROOT))
print()
for p in sorted(config.OUTPUTS_FIGURES.glob("*.png")):
    print("  ", p.relative_to(config.REPO_ROOT))